In [358]:
# Importuj wymagane biblioteki
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datasets import concatenate_datasets
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from datasets import load_from_disk
from collections import Counter, defaultdict
from scipy.stats import chi2_contingency, ks_2samp
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
pio.templates.default = "plotly_white"

In [359]:
# Wczytaj konfigurację
with open("../config.json") as f:
    cfg = json.load(f)

print(f"📋 Konfiguracja:")
print(f"   Model: {cfg['model_name']}")
print(f"   Ścieżka datasetu: {cfg.get('hf_dataset', 'N/A')}")
print(f"   Wyjście tokenizera: {cfg.get('tokenizer_output_path', 'N/A')}")

📋 Konfiguracja:
   Model: sdadas/polish-longformer-base-4096
   Ścieżka datasetu: data/processed/hf_dataset
   Wyjście tokenizera: data/processed/tokenized_dataset


In [360]:
# Wczytaj stokenizowany dataset
print("\n📂 Wczytywanie stokenizowanego datasetu...")

dataset = load_from_disk("../" + cfg["tokenizer_output_path"])
# Check if dataset has train/validation splits
if hasattr(dataset, 'keys') and 'train' in dataset and 'validation' in dataset:
    print(f"✅ Wczytano dataset z podziałem train/validation:")
    print(f"   📚 Train: {len(dataset['train'])} fragmentów")
    print(f"   📖 Validation: {len(dataset['validation'])} fragmentów")
    print(f"   📊 Łącznie: {len(dataset['train']) + len(dataset['validation'])} fragmentów")
    
    # Display features
    print("\n📋 Klucze cech datasetu:")
    for key in dataset['train'].features.keys():
        print(f"   - {key}")
    print(f"\nLiczba cech: {len(dataset['train'].features)}")
    
    # Use combined dataset for overall analysis, but keep splits for comparison
    combined_dataset = concatenate_datasets([dataset['train'], dataset['validation']])
    train_dataset = dataset['train']
    val_dataset = dataset['validation']
    has_splits = True
else:
    print(f"✅ Wczytano dataset (pojedynczy) z {len(dataset)} fragmentami")
    print("⚠️  Dataset nie ma podziału train/validation - używając całego datasetu")
    
    # Display features
    print("📋 Klucze cech datasetu:")
    for key in dataset.features.keys():
        print(f"   - {key}")
    print(f"\nLiczba cech: {len(dataset.features)}")
    
    combined_dataset = dataset
    train_dataset = None
    val_dataset = None
    has_splits = False



📂 Wczytywanie stokenizowanego datasetu...
✅ Wczytano dataset z podziałem train/validation:
   📚 Train: 5779 fragmentów
   📖 Validation: 1282 fragmentów
   📊 Łącznie: 7061 fragmentów

📋 Klucze cech datasetu:
   - labels
   - doc_id
   - input_ids
   - token_type_ids
   - attention_mask

Liczba cech: 5


In [361]:
# Wyodrębnij podstawowe informacje
doc_ids = combined_dataset['doc_id']
labels = np.array(combined_dataset['labels'])
unique_docs = len(set(doc_ids))
total_chunks = len(combined_dataset)

print(f"📈 Przegląd łącznego datasetu:")
print(f"   Łączna liczba dokumentów: {unique_docs:,}")
print(f"   Łączna liczba fragmentów: {total_chunks:,}")
print(f"   Średnia liczba fragmentów na dokument: {total_chunks/unique_docs:.2f}")
print(f"   Liczba etykiet: {labels.shape[1]}")

if has_splits:
    print(f"\n📊 Szczegóły podziału train/validation:")
    
    # Train split analysis
    train_doc_ids = train_dataset['doc_id']
    train_labels = np.array(train_dataset['labels'])
    train_unique_docs = len(set(train_doc_ids))
    train_chunks = len(train_dataset)
    
    print(f"   🚂 TRAIN:")
    print(f"      Dokumenty: {train_unique_docs:,} ({train_unique_docs/unique_docs*100:.1f}%)")
    print(f"      Fragmenty: {train_chunks:,} ({train_chunks/total_chunks*100:.1f}%)")
    print(f"      Średnia fragmentów/dok: {train_chunks/train_unique_docs:.2f}")
    
    # Validation split analysis
    val_doc_ids = val_dataset['doc_id']
    val_labels = np.array(val_dataset['labels'])
    val_unique_docs = len(set(val_doc_ids))
    val_chunks = len(val_dataset)
    
    print(f"   🔍 VALIDATION:")
    print(f"      Dokumenty: {val_unique_docs:,} ({val_unique_docs/unique_docs*100:.1f}%)")
    print(f"      Fragmenty: {val_chunks:,} ({val_chunks/total_chunks*100:.1f}%)")
    print(f"      Średnia fragmentów/dok: {val_chunks/val_unique_docs:.2f}")
    
    # Check for doc_id overlap (should be none)
    train_doc_set = set(train_doc_ids)
    val_doc_set = set(val_doc_ids)
    overlap = train_doc_set.intersection(val_doc_set)
    
    if overlap:
        print(f"   ⚠️  UWAGA: {len(overlap)} dokumentów występuje w obu zbiorach!")
    else:
        print(f"   ✅ Brak nakładania się dokumentów między zbiorami")
else:
    print("\n   ℹ️  Analiza podziału nie jest dostępna (brak train/validation split)")

📈 Przegląd łącznego datasetu:
   Łączna liczba dokumentów: 393
   Łączna liczba fragmentów: 7,061
   Średnia liczba fragmentów na dokument: 17.97
   Liczba etykiet: 7

📊 Szczegóły podziału train/validation:
   🚂 TRAIN:
      Dokumenty: 314 (79.9%)
      Fragmenty: 5,779 (81.8%)
      Średnia fragmentów/dok: 18.40
   🔍 VALIDATION:
      Dokumenty: 79 (20.1%)
      Fragmenty: 1,282 (18.2%)
      Średnia fragmentów/dok: 16.23
   ✅ Brak nakładania się dokumentów między zbiorami


In [362]:
# Etykiety kryteriów weryfikacji
ESG_LABELS = ['C1', 'C2', 'C3', 'C5', 'C8', 'C9', 'C10']

In [363]:
doc_chunk_counts = Counter(doc_ids)
chunk_counts = list(doc_chunk_counts.values())

print(f"\n📏 Rozkład długości dokumentów (w fragmentach):")
print(f"   Min fragmentów na dokument: {min(chunk_counts)}")
print(f"   Max fragmentów na dokument: {max(chunk_counts)}")
print(f"   Mediana fragmentów na dokument: {np.median(chunk_counts):.1f}")
print(f"   Średnia fragmentów na dokument: {np.mean(chunk_counts):.2f}")
print(f"   Odch. std. fragmentów na dokument: {np.std(chunk_counts):.2f}")


📏 Rozkład długości dokumentów (w fragmentach):
   Min fragmentów na dokument: 1
   Max fragmentów na dokument: 77
   Mediana fragmentów na dokument: 15.0
   Średnia fragmentów na dokument: 17.97
   Odch. std. fragmentów na dokument: 12.04


In [364]:
# Utwórz DataFrame o długości dokumentu, aby ułatwić tworzenie wykresów
length_df = pd.DataFrame({
    'doc_id': list(doc_chunk_counts.keys()),
    'chunk_count': list(doc_chunk_counts.values())
})

In [ ]:
# Create subplots for document length analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Rozkład fragmentów na dokument',
        'Rozkład liczby fragmentów',
        'Rozkład skumulowany długości dokumentów',
        'Kategorie długości dokumentów'
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"type": "pie"}]]
)

In [366]:
# Histogram of chunk counts
fig.add_trace(
    go.Histogram(
        x=chunk_counts,
        nbinsx=30,
        name="Chunk Distribution",
        opacity=0.7,
        marker_color="steelblue"
    ),
    row=1, col=1
)

# Add mean line
mean_chunks = np.mean(chunk_counts)
fig.add_vline(
    x=mean_chunks,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Średnia: {mean_chunks:.1f}",
    row=1, col=1
)

# Box plot
fig.add_trace(
    go.Box(
        y=chunk_counts,
        name="Liczenie fragmentów",
        marker_color="lightblue"
    ),
    row=1, col=2
)

# Cumulative distribution
sorted_counts = np.sort(chunk_counts)
cumulative = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)

fig.add_trace(
    go.Scatter(
        x=sorted_counts,
        y=cumulative,
        mode='lines',
        name='Łączna dystrybucja',
        line=dict(width=3, color='green')
    ),
    row=2, col=1
)

# Length categories pie chart
length_categories = pd.cut(chunk_counts, 
                          bins=[0, 5, 10, 20, 50, float('inf')], 
                          labels=['Bardzo krótki (1-5)', 'Krótki (6-10)', 'Średni (11-20)', 
                                  'Długi (21-50)', 'Bardzo długi (50+)'])
category_counts = length_categories.value_counts()

fig.add_trace(
    go.Pie(
        labels=category_counts.index,
        values=category_counts.values,
        name="Kategorie długości"
    ),
    row=2, col=2
)

# Update layout
fig.update_layout(
    height=800,
    width=1000,
    title_text="Analiza długości dokumentu",
    title_x=0.5,
    title_font_size=20,
    showlegend=False
)

# Update axis labels
fig.update_xaxes(title_text="Liczba dokumentów", row=1, col=1)
fig.update_yaxes(title_text="Liczba fragmentów", row=1, col=2)
fig.update_xaxes(title_text="Liczba fragmentów", row=2, col=1)
fig.update_yaxes(title_text="Prawdopodobieństwo skumulowane", row=2, col=1)

fig.show()

In [367]:
# Wypisz statystyki długości
print(f"📊 Podział kategorii długości:")
for category, count in category_counts.items():
    percentage = (count / len(chunk_counts)) * 100
    print(f"   {category}: {count} dokumentów ({percentage:.1f}%)")

📊 Podział kategorii długości:
   Bardzo krótki (1-5): 26 dokumentów (6.6%)
   Krótki (6-10): 85 dokumentów (21.6%)
   Średni (11-20): 160 dokumentów (40.7%)
   Długi (21-50): 112 dokumentów (28.5%)
   Bardzo długi (50+): 10 dokumentów (2.5%)


In [ ]:
# Porównanie podziałów train/validation
if has_splits:
    print(f"\n🔍 ANALIZA PODZIAŁU TRAIN/VALIDATION")
    print("="*50)
    
    # Document length comparison between splits
    train_doc_chunk_counts = Counter(train_doc_ids)
    val_doc_chunk_counts = Counter(val_doc_ids)
    
    train_chunk_counts = list(train_doc_chunk_counts.values())
    val_chunk_counts = list(val_doc_chunk_counts.values())
    
    print(f"\n📏 Porównanie długości dokumentów:")
    print(f"   TRAIN - Min: {min(train_chunk_counts)}, Max: {max(train_chunk_counts)}, Średnia: {np.mean(train_chunk_counts):.2f}")
    print(f"   VAL   - Min: {min(val_chunk_counts)}, Max: {max(val_chunk_counts)}, Średnia: {np.mean(val_chunk_counts):.2f}")
    
    # Label distribution comparison
    train_doc_labels = defaultdict(list)
    val_doc_labels = defaultdict(list)
    
    for i, doc_id in enumerate(train_doc_ids):
        train_doc_labels[doc_id].append(train_labels[i])
    
    for i, doc_id in enumerate(val_doc_ids):
        val_doc_labels[doc_id].append(val_labels[i])
    
    # Convert to document-level labels using majority voting
    train_doc_level_labels = []
    for doc_id in sorted(train_doc_labels.keys()):
        chunk_labels = np.array(train_doc_labels[doc_id])
        doc_label = (np.mean(chunk_labels, axis=0) >= 0.5).astype(int)
        train_doc_level_labels.append(doc_label)
    
    val_doc_level_labels = []
    for doc_id in sorted(val_doc_labels.keys()):
        chunk_labels = np.array(val_doc_labels[doc_id])
        doc_label = (np.mean(chunk_labels, axis=0) >= 0.5).astype(int)
        val_doc_level_labels.append(doc_label)
    
    train_doc_level_labels = np.array(train_doc_level_labels)
    val_doc_level_labels = np.array(val_doc_level_labels)
    
    # Calculate label frequencies for each split
    train_label_frequencies = np.sum(train_doc_level_labels, axis=0)
    val_label_frequencies = np.sum(val_doc_level_labels, axis=0)
    
    print(f"\n🏷️  Porównanie rozkładu etykiet ESG:")
    print(f"{'Etykieta':<8} {'Train %':<10} {'Val %':<10} {'Różnica':<10}")
    print("-" * 40)
    
    for i, label_name in enumerate(ESG_LABELS):
        if i < len(train_label_frequencies):
            train_pct = (train_label_frequencies[i] / len(train_doc_level_labels)) * 100
            val_pct = (val_label_frequencies[i] / len(val_doc_level_labels)) * 100
            diff = abs(train_pct - val_pct)
            print(f"{label_name:<8} {train_pct:<10.1f} {val_pct:<10.1f} {diff:<10.1f}")
    

    # Create contingency table for each label
    print(f"\n📊 Test zgodności rozkładu (Chi-square):")
    for i, label_name in enumerate(ESG_LABELS):
        if i < len(train_label_frequencies):
            contingency_table = [
                [train_label_frequencies[i], len(train_doc_level_labels) - train_label_frequencies[i]],
                [val_label_frequencies[i], len(val_doc_level_labels) - val_label_frequencies[i]]
            ]
            
            try:
                chi2, p_value, _, _ = chi2_contingency(contingency_table)
                status = "✅ Podobne" if p_value > 0.05 else "⚠️ Różne"
                print(f"   {label_name}: p={p_value:.3f} {status}")
            except:
                print(f"   {label_name}: Nie można obliczyć (za mało danych)")
else:
    print("\n   ℹ️  Analiza podziału train/validation nie jest dostępna")


🔍 ANALIZA PODZIAŁU TRAIN/VALIDATION

📏 Porównanie długości dokumentów:
   TRAIN - Min: 1, Max: 77, Średnia: 18.40
   VAL   - Min: 2, Max: 56, Średnia: 16.23

🏷️  Porównanie rozkładu etykiet ESG:
Etykieta Train %    Val %      Różnica   
----------------------------------------
C1       59.9       55.7       4.2       
C2       65.0       54.4       10.5      
C3       29.9       22.8       7.2       
C5       57.0       53.2       3.8       
C8       51.0       55.7       4.7       
C9       39.8       35.4       4.4       
C10      56.4       59.5       3.1       

📊 Test zgodności rozkładu (Chi-square):
   C1: p=0.585 ✅ Podobne
   C2: p=0.109 ✅ Podobne
   C3: p=0.263 ✅ Podobne
   C5: p=0.625 ✅ Podobne
   C8: p=0.530 ✅ Podobne
   C9: p=0.560 ✅ Podobne
   C10: p=0.708 ✅ Podobne


In [ ]:
# Wizualizacja porównania train/validation
if has_splits:
    # Create comparison visualization
    fig_comparison = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Rozkład długości dokumentów - porównanie',
            'Rozkład etykiet ESG - porównanie',
            'Fragmenty na dokument - wykres pudełkowy',
            'Podsumowanie statystyk'
        ],
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"type": "table"}]]
    )
    
    # Document length comparison histogram
    fig_comparison.add_trace(
        go.Histogram(
            x=train_chunk_counts,
            name="Train",
            opacity=0.7,
            marker_color="blue",
            nbinsx=20
        ),
        row=1, col=1
    )
    
    fig_comparison.add_trace(
        go.Histogram(
            x=val_chunk_counts,
            name="Validation",
            opacity=0.7,
            marker_color="red",
            nbinsx=20
        ),
        row=1, col=1
    )
    
    # ESG label frequency comparison
    label_names = [ESG_LABELS[i] if i < len(ESG_LABELS) else f'Label_{i}' for i in range(len(train_label_frequencies))]
    
    fig_comparison.add_trace(
        go.Bar(
            x=label_names,
            y=train_label_frequencies / len(train_doc_level_labels) * 100,
            name="Train %",
            marker_color="blue",
            opacity=0.7
        ),
        row=1, col=2
    )
    
    fig_comparison.add_trace(
        go.Bar(
            x=label_names,
            y=val_label_frequencies / len(val_doc_level_labels) * 100,
            name="Validation %",
            marker_color="red",
            opacity=0.7
        ),
        row=1, col=2
    )
    
    # Box plot comparison for document lengths
    fig_comparison.add_trace(
        go.Box(
            y=train_chunk_counts,
            name="Train",
            marker_color="blue"
        ),
        row=2, col=1
    )
    
    fig_comparison.add_trace(
        go.Box(
            y=val_chunk_counts,
            name="Validation",
            marker_color="red"
        ),
        row=2, col=1
    )
    
    # Summary statistics table
    split_stats = [
        ['Metric', 'Train', 'Validation', 'Różnica'],
        ['Dokumenty', f'{len(train_doc_level_labels)}', f'{len(val_doc_level_labels)}', f'{abs(len(train_doc_level_labels) - len(val_doc_level_labels))}'],
        ['Fragmenty', f'{len(train_dataset)}', f'{len(val_dataset)}', f'{abs(len(train_dataset) - len(val_dataset))}'],
        ['Śr. długość dok.', f'{np.mean(train_chunk_counts):.1f}', f'{np.mean(val_chunk_counts):.1f}', f'{abs(np.mean(train_chunk_counts) - np.mean(val_chunk_counts)):.1f}'],
        ['Med. długość dok.', f'{np.median(train_chunk_counts):.1f}', f'{np.median(val_chunk_counts):.1f}', f'{abs(np.median(train_chunk_counts) - np.median(val_chunk_counts)):.1f}'],
        ['Śr. etykiet/dok.', f'{np.mean(np.sum(train_doc_level_labels, axis=1)):.1f}', f'{np.mean(np.sum(val_doc_level_labels, axis=1)):.1f}', f'{abs(np.mean(np.sum(train_doc_level_labels, axis=1)) - np.mean(np.sum(val_doc_level_labels, axis=1))):.1f}']
    ]
    
    fig_comparison.add_trace(
        go.Table(
            header=dict(values=split_stats[0], fill_color='lightblue', align='left'),
            cells=dict(values=list(zip(*split_stats[1:])), fill_color='white', align='left')
        ),
        row=2, col=2
    )
    
    fig_comparison.update_layout(
        height=800,
        width=1000,
        title_text="Porównanie Train vs Validation Splits",
        title_x=0.5,
        title_font_size=18
    )
    
    # Update axis labels
    fig_comparison.update_xaxes(title_text="Liczba fragmentów", row=1, col=1)
    fig_comparison.update_yaxes(title_text="Liczba dokumentów", row=1, col=1)
    fig_comparison.update_yaxes(title_text="Procent dokumentów", row=1, col=2)
    fig_comparison.update_yaxes(title_text="Liczba fragmentów", row=2, col=1)
    
    fig_comparison.show()
    
    # Calculate and display split quality metrics
    print(f"\n🎯 OCENA JAKOŚCI PODZIAŁU:")
    
    # Document length distribution similarity
    ks_stat, ks_p = ks_2samp(train_chunk_counts, val_chunk_counts)
    print(f"   📏 Długości dokumentów: KS test p={ks_p:.3f} {'✅ Podobne' if ks_p > 0.05 else '⚠️ Różne'}")
    
    # Label distribution similarity (overall)
    train_label_props = train_label_frequencies / len(train_doc_level_labels)
    val_label_props = val_label_frequencies / len(val_doc_level_labels)
    max_diff = np.max(np.abs(train_label_props - val_label_props))
    print(f"   🏷️  Rozkład etykiet: Max różnica {max_diff:.3f} {'✅ Dobry' if max_diff < 0.05 else '⚠️ Sprawdź' if max_diff < 0.1 else '❌ Problematyczny'}")
    
    # Size balance
    size_ratio = min(len(train_dataset), len(val_dataset)) / max(len(train_dataset), len(val_dataset))
    print(f"   📊 Balans wielkości: {size_ratio:.3f} {'✅ Zbalansowany' if size_ratio > 0.7 else '⚠️ Niezbalansowany'}")
    
    print(f"\n   🎓 Rekomendacje dla eksperymentów:")
    if ks_p > 0.05 and max_diff < 0.05 and size_ratio > 0.7:
        print(f"      ✅ Podział jest odpowiedni do treningu modelu")
    else:
        if ks_p <= 0.05:
            print(f"      ⚠️ Rozważ stratyfikację według długości dokumentów")
        if max_diff >= 0.05:
            print(f"      ⚠️ Rozważ stratyfikację według rozkładu etykiet")
        if size_ratio <= 0.7:
            print(f"      ⚠️ Rozważ przebalansowanie wielkości zbiorów")
            
else:
    print("\n   📊 Brak analizy porównawczej - dataset nie ma podziału train/validation")


🎯 OCENA JAKOŚCI PODZIAŁU:
   📏 Długości dokumentów: KS test p=0.235 ✅ Podobne
   🏷️  Rozkład etykiet: Max różnica 0.105 ❌ Problematyczny
   📊 Balans wielkości: 0.222 ⚠️ Niezbalansowany

   🎓 Rekomendacje dla eksperymentów:
      ⚠️ Rozważ stratyfikację według rozkładu etykiet
      ⚠️ Rozważ przebalansowanie wielkości zbiorów


In [370]:
# Zagregowane etykiety na poziomie dokumentu (głosowanie większościowe)
doc_labels = defaultdict(list)
for i, doc_id in enumerate(doc_ids):
    doc_labels[doc_id].append(labels[i])

In [371]:
# Konwersja na etykiety na poziomie dokumentu przy użyciu większości głosów
doc_level_labels = []
for doc_id in sorted(doc_labels.keys()):
    chunk_labels = np.array(doc_labels[doc_id])
    # Użyj głosowania większościowego (>= 50% fragmentów musi mieć etykietę=1)
    doc_label = (np.mean(chunk_labels, axis=0) >= 0.5).astype(int)
    doc_level_labels.append(doc_label)

doc_level_labels = np.array(doc_level_labels)

print(f"📈 Statystyki etykiet:")
label_frequencies = np.sum(doc_level_labels, axis=0)
total_docs = len(doc_level_labels)

for i, freq in enumerate(label_frequencies):
    percentage = (freq / total_docs) * 100
    label_name = ESG_LABELS[i] if i < len(ESG_LABELS) else f"Label_{i}"
    print(f"   {label_name}: {freq}/{total_docs} ({percentage:.1f}%)")

📈 Statystyki etykiet:
   C1: 232/393 (59.0%)
   C2: 247/393 (62.8%)
   C3: 112/393 (28.5%)
   C5: 221/393 (56.2%)
   C8: 204/393 (51.9%)
   C9: 153/393 (38.9%)
   C10: 224/393 (57.0%)


In [372]:
# Analiza współwystępowania etykiet ESG
label_combinations = []
for doc_labels in doc_level_labels:
    active_labels = [i for i, label in enumerate(doc_labels) if label == 1]
    if active_labels:
        label_combinations.append(tuple(sorted(active_labels)))

combo_counter = Counter(label_combinations)
print(f"Najczęstsze kombinacje etykiet:")
for combo, count in combo_counter.most_common(10):
    percentage = (count / total_docs) * 100
    if combo:
        labels_str = ', '.join([ESG_LABELS[i] if i < len(ESG_LABELS) else f"Label_{i}" for i in combo])
    else:
        labels_str = 'Brak etykiet'
    print(f"   {labels_str}: {count} dokumentów ({percentage:.1f}%)")

Najczęstsze kombinacje etykiet:
   C1, C2, C3, C5, C8, C9, C10: 55 dokumentów (14.0%)
   C1, C2, C5, C8, C9, C10: 33 dokumentów (8.4%)
   C1, C2, C5, C8, C10: 19 dokumentów (4.8%)
   C1, C2, C3, C5, C8, C10: 19 dokumentów (4.8%)
   C2: 18 dokumentów (4.6%)
   C1: 16 dokumentów (4.1%)
   C1, C2: 9 dokumentów (2.3%)
   C10: 7 dokumentów (1.8%)
   C2, C10: 7 dokumentów (1.8%)
   C1, C10: 6 dokumentów (1.5%)


In [373]:
# Create label analysis subplots
fig_labels = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
      'Częstotliwości etykiet (poziom dokumentu)',
      'Macierz korelacji etykiet',
      'Liczba etykiet na dokument',
      'Proporcja etykiet (przypadków pozytywnych)'
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

In [374]:
# Label frequency bar plot with ESG names
label_names = [ESG_LABELS[i] if i < len(ESG_LABELS) else f'Label_{i}' for i in range(len(label_frequencies))]
fig_labels.add_trace(
    go.Bar(
        x=label_names,
        y=label_frequencies,
        name="Częstotliwości etykiet",
        marker_color="steelblue"
    ),
    row=1, col=1
)

# Label correlation heatmap
label_corr = np.corrcoef(doc_level_labels.T)
fig_labels.add_trace(
    go.Heatmap(
        z=label_corr,
        x=label_names,
        y=label_names,
        colorscale='RdBu',
        zmid=0,
        showscale=True,
        colorbar=dict(x=0.45)
    ),
    row=1, col=2
)

# Number of labels per document
labels_per_doc = np.sum(doc_level_labels, axis=1)
fig_labels.add_trace(
    go.Histogram(
        x=labels_per_doc,
        nbinsx=int(max(labels_per_doc)+1),
        name="Etykiety na dokument",
        marker_color="lightcoral"
    ),
    row=2, col=1
)

# Label balance visualization
label_balance = label_frequencies / total_docs
fig_labels.add_trace(
    go.Bar(
        x=label_names,
        y=label_balance,
        name="Proporcja etykiet",
        marker_color="lightgreen"
    ),
    row=2, col=2
)

# Add balanced line (50%)
fig_labels.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="red",
    annotation_text="Wyśrodkowany (50%)",
    row=2, col=2
)

# Update layout for label analysis
fig_labels.update_layout(
    height=800,
    title_text="Analiza etykiet",
    title_x=0.5,
    title_font_size=20,
    showlegend=False
)

# Update axis labels
fig_labels.update_yaxes(title_text="Liczba dokumentów", row=1, col=1)
fig_labels.update_xaxes(title_text="Ilość aktywnych etykiet", row=2, col=1)
fig_labels.update_yaxes(title_text="Liczba dokumentów", row=2, col=1)
fig_labels.update_yaxes(title_text="Proporcja", row=2, col=2)

fig_labels.show()

In [375]:
# Podsumowanie analizy etykiet
most_freq_idx = np.argmax(label_frequencies)
least_freq_idx = np.argmin(label_frequencies)
most_freq_label = ESG_LABELS[most_freq_idx] if most_freq_idx < len(ESG_LABELS) else f"Label_{most_freq_idx}"
least_freq_label = ESG_LABELS[least_freq_idx] if least_freq_idx < len(ESG_LABELS) else f"Label_{least_freq_idx}"

print(f"   Najczęściej występujące etykiety: {most_freq_label} ({max(label_frequencies)} docs, {max(label_frequencies)/total_docs*100:.1f}%)")
print(f"   Najrzadziej występujące etykiety: {least_freq_label} ({min(label_frequencies)} docs, {min(label_frequencies)/total_docs*100:.1f}%)")
print(f"   Średnia ilość etykiet na dokument: {np.mean(labels_per_doc):.2f}")
print(f"   Dokumenty bez etykiet: {sum(labels_per_doc == 0)}")
print(f"   Dokumenty z wszystkimi etykietami: {sum(labels_per_doc == len(label_frequencies))}")

print(f"\n🎯 Spostrzeżenia dotyczące klasyfikacji:")
for i, (label_name, freq) in enumerate(zip(ESG_LABELS, label_frequencies)):
    if i < len(label_frequencies):
        percentage = (freq / total_docs) * 100
        if percentage > 50:
            print(f"   ✓ {label_name} jest dobrze reprezentowana ({percentage:.1f}%)")
        elif percentage > 20:
            print(f"   ○ {label_name} jest umiarkowanie reprezentowana ({percentage:.1f}%)")
        else:
            print(f"   ⚠ {label_name} jest niedoreprezentowana ({percentage:.1f}%)")

   Najczęściej występujące etykiety: C2 (247 docs, 62.8%)
   Najrzadziej występujące etykiety: C3 (112 docs, 28.5%)
   Średnia ilość etykiet na dokument: 3.54
   Dokumenty bez etykiet: 70
   Dokumenty z wszystkimi etykietami: 55

🎯 Spostrzeżenia dotyczące klasyfikacji:
   ✓ C1 jest dobrze reprezentowana (59.0%)
   ✓ C2 jest dobrze reprezentowana (62.8%)
   ○ C3 jest umiarkowanie reprezentowana (28.5%)
   ✓ C5 jest dobrze reprezentowana (56.2%)
   ✓ C8 jest dobrze reprezentowana (51.9%)
   ○ C9 jest umiarkowanie reprezentowana (38.9%)
   ✓ C10 jest dobrze reprezentowana (57.0%)


In [376]:
# Utwórz dataset łączący długość dokumentów i etykiety
doc_analysis_data = []
doc_labels_dict = defaultdict(list)
for i, doc_id_val in enumerate(doc_ids):
    doc_labels_dict[doc_id_val].append(labels[i])

for doc_id in set(doc_ids):
    doc_chunk_count = doc_chunk_counts[doc_id]
    doc_label_idx = list(sorted(doc_labels_dict.keys())).index(doc_id)
    doc_label_vector = doc_level_labels[doc_label_idx]
    
    record = {
        'doc_id': doc_id,
        'chunk_count': doc_chunk_count,
        'total_labels': sum(doc_label_vector),
        'length_category': 'Krótkie' if doc_chunk_count <= 10 else 'Umiarkowane' if doc_chunk_count <= 20 else 'Długie'
    }
    
    for i, label_name in enumerate(ESG_LABELS):
        if i < len(doc_label_vector):
            record[f'has_{label_name}'] = doc_label_vector[i]
    
    doc_analysis_data.append(record)

doc_df = pd.DataFrame(doc_analysis_data)

# Display ESG label distribution by document length
print(f"📊 Etykiety ESG według kategorii długości dokumentu:")
for category in ['Krótkie', 'Umiarkowane', 'Długie']:
    category_docs = doc_df[doc_df['length_category'] == category]
    if len(category_docs) > 0:
        print(f"\n   {category} dokumenty ({len(category_docs)} dokumenty):")
        for label_name in ESG_LABELS:
            col_name = f'has_{label_name}'
            if col_name in category_docs.columns:
                count = category_docs[col_name].sum()
                percentage = (count / len(category_docs)) * 100
                print(f"      {label_name}: {count}/{len(category_docs)} ({percentage:.1f}%)")

📊 Etykiety ESG według kategorii długości dokumentu:

   Krótkie dokumenty (111 dokumenty):
      C1: 48/111 (43.2%)
      C2: 50/111 (45.0%)
      C3: 13/111 (11.7%)
      C5: 41/111 (36.9%)
      C8: 31/111 (27.9%)
      C9: 20/111 (18.0%)
      C10: 39/111 (35.1%)

   Umiarkowane dokumenty (160 dokumenty):
      C1: 85/160 (53.1%)
      C2: 96/160 (60.0%)
      C3: 38/160 (23.8%)
      C5: 90/160 (56.2%)
      C8: 87/160 (54.4%)
      C9: 72/160 (45.0%)
      C10: 89/160 (55.6%)

   Długie dokumenty (122 dokumenty):
      C1: 99/122 (81.1%)
      C2: 101/122 (82.8%)
      C3: 61/122 (50.0%)
      C5: 90/122 (73.8%)
      C8: 86/122 (70.5%)
      C9: 61/122 (50.0%)
      C10: 96/122 (78.7%)


In [377]:
# Interaktywny wykres punktowy: Długość dokumentu a liczba etykiet
fig_scatter = px.scatter(
    doc_df,
    x='chunk_count',
    y='total_labels',
    color='length_category',
    size='chunk_count',
    hover_data=['doc_id'],
    title='Długość dokumentu a liczba etykiet',
    labels={
    'chunk_count': 'Liczba fragmentów',
    'total_labels': 'Liczba aktywnych etykiet',
    'length_category': 'Kategoria długości'
    },
    color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c']
)

fig_scatter.update_layout(height=600, title_x=0.5)
fig_scatter.show()

In [378]:
# Wykres pudełkowy: Rozkład etykiet według kategorii długości dokumentu
fig_box = px.box(
    doc_df,
    x='length_category',
    y='total_labels',
    title='Rozkład etykiet według kategorii długości dokumentu',
    labels={
    'length_category': 'Kategoria długości dokumentu',
    'total_labels': 'Liczba aktywnych etykiet'
    }
)

fig_box.update_layout(height=500, title_x=0.5)
fig_box.show()

In [379]:
# Analiza, jak fragmentacja wpływa na spójność etykiet
chunk_level_labels = labels
print(f"📊 Spójność etykiet fragmentów i dokumentów:")
inconsistent_docs = 0
consistency_scores = []
consistency_data = []

for doc_id in set(doc_ids):
    doc_chunk_indices = [i for i, d_id in enumerate(doc_ids) if d_id == doc_id]
    doc_chunk_labels = chunk_level_labels[doc_chunk_indices]
    
    # Calculate consistency (how often chunks agree on each label)
    if len(doc_chunk_labels) > 1:
        consistency = []
        for label_idx in range(doc_chunk_labels.shape[1]):
            label_values = doc_chunk_labels[:, label_idx]
            # Consistency = 1 if all chunks agree, 0 if maximally disagreed
            unique_values = len(np.unique(label_values))
            label_consistency = 1.0 if unique_values == 1 else 0.0
            consistency.append(label_consistency)
        
        avg_consistency = np.mean(consistency)
        consistency_scores.append(avg_consistency)
        
        consistency_data.append({
            'doc_id': doc_id,
            'chunk_count': len(doc_chunk_labels),
            'consistency_score': avg_consistency,
            'is_consistent': avg_consistency == 1.0
        })
        
        if avg_consistency < 1.0:
            inconsistent_docs += 1

consistency_df = pd.DataFrame(consistency_data)

print(f"   Documents with inconsistent chunk labels: {inconsistent_docs}/{len(set(doc_ids))} ({inconsistent_docs/len(set(doc_ids))*100:.1f}%)")
if consistency_scores:
    print(f"   Average consistency score: {np.mean(consistency_scores):.3f}")
    print(f"   Min consistency score: {min(consistency_scores):.3f}")


📊 Spójność etykiet fragmentów i dokumentów:
   Documents with inconsistent chunk labels: 0/393 (0.0%)
   Average consistency score: 1.000
   Min consistency score: 1.000


In [380]:
# Visualize consistency vs document length
if len(consistency_df) > 0:
    fig_consistency = px.scatter(
        consistency_df,
        x='chunk_count',
        y='consistency_score',
        color='is_consistent',
        title='Label Consistency vs Document Length',
        labels={
            'chunk_count': 'Number of Chunks',
            'consistency_score': 'Label Consistency Score',
            'is_consistent': 'Fully Consistent'
        },
        hover_data=['doc_id']
    )
    
    fig_consistency.update_layout(height=500, title_x=0.5)
    fig_consistency.show()

In [381]:
# Estimate computational requirements for different approaches
max_chunks_per_doc = max(chunk_counts)
avg_chunks_per_doc = np.mean(chunk_counts)

print(f"🔢 Computational Complexity Estimates:")
print(f"   Max chunks per document: {max_chunks_per_doc}")
print(f"   Average chunks per document: {avg_chunks_per_doc:.1f}")
print(f"   Total model forward passes needed:")
print(f"      Simple aggregation: {total_chunks:,} (one per chunk)")
print(f"      Hierarchical (2-stage): {total_chunks:,} + {unique_docs:,} = {total_chunks + unique_docs:,}")
print(f"      Positional weighting: {total_chunks:,} + weight computation")

🔢 Computational Complexity Estimates:
   Max chunks per document: 77
   Average chunks per document: 18.0
   Total model forward passes needed:
      Simple aggregation: 7,061 (one per chunk)
      Hierarchical (2-stage): 7,061 + 393 = 7,454
      Positional weighting: 7,061 + weight computation


In [382]:
# Memory requirements (rough estimates)
tokens_per_chunk = 4096
bytes_per_token = 4  # Assuming int32
chunk_memory_mb = (tokens_per_chunk * bytes_per_token) / (1024 * 1024)
total_memory_gb = (total_chunks * chunk_memory_mb) / 1024

print(f"\n💾 Memory Requirements (rough estimates):")
print(f"   Memory per chunk: {chunk_memory_mb:.2f} MB")
print(f"   Total dataset memory: {total_memory_gb:.2f} GB")
print(f"   Batch processing recommendations:")
print(f"      For 16GB GPU: ~{int(16000 / chunk_memory_mb)} chunks per batch")
print(f"      Document-level batching: Group by similar doc lengths")


💾 Memory Requirements (rough estimates):
   Memory per chunk: 0.02 MB
   Total dataset memory: 0.11 GB
   Batch processing recommendations:
      For 16GB GPU: ~1024000 chunks per batch
      Document-level batching: Group by similar doc lengths


In [383]:
# Create computational requirements visualization
comp_data = pd.DataFrame({
    'Approach': ['Simple Aggregation', 'Hierarchical (2-stage)', 'Positional Weighting'],
    'Forward_Passes': [total_chunks, total_chunks + unique_docs, total_chunks],
    'Additional_Computation': ['None', 'Attention Mechanism', 'Weight Calculation'],
    'Memory_Efficiency': ['Standard', 'Higher (2 models)', 'Standard']
})

fig_comp = px.bar(
    comp_data,
    x='Approach',
    y='Forward_Passes',
    title='Computational Requirements by Approach',
    labels={'Forward_Passes': 'Number of Forward Passes'},
    text='Forward_Passes'
)

fig_comp.update_traces(texttemplate='%{text:,}', textposition='outside')
fig_comp.update_layout(height=500, title_x=0.5)
fig_comp.show()


In [384]:
print("\n" + "="*60)
print("🎓 THESIS EXPERIMENT RECOMMENDATIONS")
print("="*60)

print("📋 Based on your data analysis, here are the recommendations:")

print(f"\n1️⃣  DATASET CHARACTERISTICS:")
print(f"   ✓ Dataset size is suitable for thesis experiments ({unique_docs:,} docs)")
print(f"   ✓ Variable document lengths provide good test cases")
if max(chunk_counts) > 20:
    print(f"   ⚠️  Some very long documents ({max(chunk_counts)} chunks) - consider subsampling for initial experiments")

print(f"\n2️⃣  LABEL ANALYSIS INSIGHTS:")
imbalance_ratio = max(label_frequencies) / min(label_frequencies) if min(label_frequencies) > 0 else float('inf')
if imbalance_ratio > 10:
    print(f"   ⚠️  High label imbalance (ratio: {imbalance_ratio:.1f}) - consider weighted loss functions")
else:
    print(f"   ✓ Moderate label imbalance (ratio: {imbalance_ratio:.1f}) - standard approaches should work")

if inconsistent_docs > 0:
    print(f"   📝 {inconsistent_docs} docs have inconsistent chunk labels - good test case for aggregation methods")

print(f"\n3️⃣  EXPERIMENTAL SETUP SUGGESTIONS:")
print(f"   📊 Create stratified splits based on:")
print(f"      - Document length categories")
print(f"      - Label frequency")
print(f"      - Label combinations")
print(f"   🔬 Ablation studies to conduct:")
print(f"      - Chunk size: 2048 vs 4096 vs 8192 tokens")
print(f"      - Stride: 256 vs 512 vs 1024")
print(f"      - Max chunks per document: 10, 20, 50, unlimited")

print(f"\n4️⃣  POSITIONAL WEIGHTING HYPOTHESIS:")
if np.mean(chunk_counts) > 5:
    print(f"   ✓ Documents are long enough to test positional weighting")
    print(f"   📈 Test with different weighting functions:")
    print(f"      - Gaussian (σ = doc_length/6, doc_length/4, doc_length/3)")
    print(f"      - Linear center-focused")
    print(f"      - Exponential decay from center")
else:
    print(f"   ⚠️  Documents might be too short for meaningful positional effects")

print(f"\n5️⃣  COMPUTATIONAL CONSIDERATIONS:")
if total_memory_gb > 32:
    print(f"   ⚠️  Large dataset - implement efficient batching and consider gradient accumulation")
else:
    print(f"   ✓ Dataset size manageable for standard GPU setups")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Implement baseline (simple mean aggregation)")
print(f"   2. Create train/val/test splits with stratification")
print(f"   3. Start with shortest documents for quick iteration")
print(f"   4. Gradually increase complexity (aggregation → hierarchical → positional)")
print(f"   5. Document all hyperparameters and results for thesis")


🎓 THESIS EXPERIMENT RECOMMENDATIONS
📋 Based on your data analysis, here are the recommendations:

1️⃣  DATASET CHARACTERISTICS:
   ✓ Dataset size is suitable for thesis experiments (393 docs)
   ✓ Variable document lengths provide good test cases
   ⚠️  Some very long documents (77 chunks) - consider subsampling for initial experiments

2️⃣  LABEL ANALYSIS INSIGHTS:
   ✓ Moderate label imbalance (ratio: 2.2) - standard approaches should work

3️⃣  EXPERIMENTAL SETUP SUGGESTIONS:
   📊 Create stratified splits based on:
      - Document length categories
      - Label frequency
      - Label combinations
   🔬 Ablation studies to conduct:
      - Chunk size: 2048 vs 4096 vs 8192 tokens
      - Stride: 256 vs 512 vs 1024
      - Max chunks per document: 10, 20, 50, unlimited

4️⃣  POSITIONAL WEIGHTING HYPOTHESIS:
   ✓ Documents are long enough to test positional weighting
   📈 Test with different weighting functions:
      - Gaussian (σ = doc_length/6, doc_length/4, doc_length/3)
      - 

In [385]:
# Create final summary visualization with ESG insights including train/val split
most_freq_idx = np.argmax(label_frequencies)
least_freq_idx = np.argmin(label_frequencies)
most_freq_label = ESG_LABELS[most_freq_idx] if most_freq_idx < len(ESG_LABELS) else f"Label_{most_freq_idx}"
least_freq_label = ESG_LABELS[least_freq_idx] if least_freq_idx < len(ESG_LABELS) else f"Label_{least_freq_idx}"

# Base metrics
summary_metrics = {
    'Metric': [
        'Total Documents', 'Total Chunks', 'Avg Chunks/Doc', 
        f'Most Frequent ESG Label ({most_freq_label})', 
        f'Least Frequent ESG Label ({least_freq_label})', 
        'Inconsistent Documents %', 'Memory Requirements (GB)',
        'ESG Categories Analyzed', 'Avg ESG Labels per Doc'
    ],
    'Value': [
        unique_docs, total_chunks, f"{avg_chunks_per_doc:.1f}",
        f"{max(label_frequencies)/total_docs*100:.1f}%",
        f"{min(label_frequencies)/total_docs*100:.1f}%",
        f"{inconsistent_docs/len(set(doc_ids))*100:.1f}%",
        f"{total_memory_gb:.1f}",
        len(ESG_LABELS),
        f"{np.mean(labels_per_doc):.2f}"
    ]
}

# Add train/validation specific metrics if available
if has_splits:
    summary_metrics['Metric'].extend([
        '--- TRAIN/VALIDATION SPLIT ---',
        'Train Documents', 'Validation Documents',
        'Train Chunks', 'Validation Chunks',
        'Train/Val Size Ratio', 'Max Label Distribution Difference',
        'Split Quality Assessment'
    ])
    
    # Calculate split-specific metrics
    split_size_ratio = min(len(train_dataset), len(val_dataset)) / max(len(train_dataset), len(val_dataset))
    train_label_props = train_label_frequencies / len(train_doc_level_labels)
    val_label_props = val_label_frequencies / len(val_doc_level_labels)
    max_label_diff = np.max(np.abs(train_label_props - val_label_props))
    
    # Assess split quality
    if split_size_ratio > 0.7 and max_label_diff < 0.05:
        split_quality = "✅ Excellent"
    elif split_size_ratio > 0.6 and max_label_diff < 0.1:
        split_quality = "🟡 Good"
    else:
        split_quality = "⚠️ Needs Review"
    
    summary_metrics['Value'].extend([
        '---',
        f"{len(train_doc_level_labels)} ({len(train_doc_level_labels)/unique_docs*100:.1f}%)",
        f"{len(val_doc_level_labels)} ({len(val_doc_level_labels)/unique_docs*100:.1f}%)",
        f"{len(train_dataset)} ({len(train_dataset)/total_chunks*100:.1f}%)",
        f"{len(val_dataset)} ({len(val_dataset)/total_chunks*100:.1f}%)",
        f"{split_size_ratio:.3f}",
        f"{max_label_diff:.3f}",
        split_quality
    ])

summary_df = pd.DataFrame(summary_metrics)

fig_summary = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>ESG Dataset Metric</b>', '<b>Value</b>'],
        fill_color='lightblue',
        align='left',
        font=dict(size=14)
    ),
    cells=dict(
        values=[summary_df.Metric, summary_df.Value],
        fill_color='white',
        align='left',
        font=dict(size=12)
    )
)])

fig_summary.update_layout(
    title="ESG Dataset Summary for Thesis Planning",
    title_x=0.5,
    height=600 if has_splits else 450
)

fig_summary.show()

# Add ESG-specific label frequency table with train/val comparison
print(f"\n📋 Detailed ESG Label Distribution:")

if has_splits:
    esg_summary = pd.DataFrame({
        'ESG_Category': ESG_LABELS[:len(label_frequencies)],
        'Total_Docs': label_frequencies,
        'Total_Pct': (label_frequencies / total_docs * 100).round(1),
        'Train_Docs': train_label_frequencies,
        'Train_Pct': (train_label_frequencies / len(train_doc_level_labels) * 100).round(1),
        'Val_Docs': val_label_frequencies,
        'Val_Pct': (val_label_frequencies / len(val_doc_level_labels) * 100).round(1),
        'Pct_Diff': np.abs((train_label_frequencies / len(train_doc_level_labels) * 100) - 
                          (val_label_frequencies / len(val_doc_level_labels) * 100)).round(1),
        'Status': ['Well-represented' if (freq/total_docs*100) > 50 
                  else 'Moderate' if (freq/total_docs*100) > 20 
                  else 'Underrepresented' 
                  for freq in label_frequencies]
    })
    
    fig_esg_table = go.Figure(data=[go.Table(
        header=dict(
            values=['<b>ESG Category</b>', '<b>Total Docs</b>', '<b>Total %</b>', 
                   '<b>Train Docs</b>', '<b>Train %</b>', '<b>Val Docs</b>', '<b>Val %</b>', 
                   '<b>% Diff</b>', '<b>Status</b>'],
            fill_color='lightgreen',
            align='left',
            font=dict(size=10)
        ),
        cells=dict(
            values=[esg_summary.ESG_Category, esg_summary.Total_Docs, 
                   [f"{p}%" for p in esg_summary.Total_Pct],
                   esg_summary.Train_Docs, [f"{p}%" for p in esg_summary.Train_Pct],
                   esg_summary.Val_Docs, [f"{p}%" for p in esg_summary.Val_Pct],
                   [f"{p}%" for p in esg_summary.Pct_Diff], esg_summary.Status],
            fill_color='white',
            align='left',
            font=dict(size=9)
        )
    ])
    
    fig_esg_table.update_layout(
        title="ESG Category Distribution Analysis (Train/Validation Comparison)",
        title_x=0.5,
        height=350
    )
else:
    esg_summary = pd.DataFrame({
        'ESG_Category': ESG_LABELS[:len(label_frequencies)],
        'Documents': label_frequencies,
        'Percentage': (label_frequencies / total_docs * 100).round(1),
        'Status': ['Well-represented' if (freq/total_docs*100) > 50 
                  else 'Moderate' if (freq/total_docs*100) > 20 
                  else 'Underrepresented' 
                  for freq in label_frequencies]
    })
    
    fig_esg_table = go.Figure(data=[go.Table(
        header=dict(
            values=['<b>ESG Category</b>', '<b>Documents</b>', '<b>Percentage</b>', '<b>Status</b>'],
            fill_color='lightgreen',
            align='left',
            font=dict(size=12)
        ),
        cells=dict(
            values=[esg_summary.ESG_Category, esg_summary.Documents, 
                   [f"{p}%" for p in esg_summary.Percentage], esg_summary.Status],
            fill_color='white',
            align='left',
            font=dict(size=11)
        )
    ])
    
    fig_esg_table.update_layout(
        title="ESG Category Distribution Analysis",
        title_x=0.5,
        height=300
    )

fig_esg_table.show()

# Final recommendations including train/val considerations
print(f"\n🎯 THESIS RECOMMENDATIONS (Updated with Train/Val Analysis):")
print("="*60)

if has_splits:
    print(f"✅ TRAIN/VALIDATION SPLIT STATUS:")
    if split_size_ratio > 0.7 and max_label_diff < 0.05:
        print(f"   🎯 Split is excellent for training - proceed with confidence")
    elif split_size_ratio > 0.6 and max_label_diff < 0.1:
        print(f"   🟡 Split is good but monitor performance carefully")
        print(f"   📝 Consider stratified validation for final model evaluation")
    else:
        print(f"   ⚠️ Split needs review:")
        if split_size_ratio <= 0.6:
            print(f"      - Size imbalance: {split_size_ratio:.3f} ratio")
        if max_label_diff >= 0.1:
            print(f"      - Label distribution difference: {max_label_diff:.3f}")
        print(f"   🔧 Recommendation: Re-run dataset creation with stratification")
        
    print(f"\n📊 EXPERIMENTAL DESIGN CONSIDERATIONS:")
    print(f"   - Use current split for initial experiments")
    print(f"   - Monitor validation performance for each ESG category")
    print(f"   - Consider class weights for imbalanced labels")
    
else:
    print(f"⚠️ NO TRAIN/VALIDATION SPLIT DETECTED:")
    print(f"   🔧 Re-run tokenization with updated script to create proper splits")
    print(f"   📝 Current analysis based on combined dataset only")

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' on line 107 (3987108122.py, line 126)